In [ ]:
!pip install datasets deep-translator transformers sentencepiece sacremoses -q

In [ ]:
!git clone https://github.com/knguyencas/ripple_nlp.git
import os
os.chdir('/kaggle/working/ripple_nlp')

In [ ]:
from datasets import load_dataset
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
import numpy as np

In [ ]:
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
label_names = ds['train'].features['labels'].feature.names
id2label = {i: name for i, name in enumerate(label_names)}

print(f"Train: {len(ds['train'])} samples")

In [ ]:
GO_EMOTION_MAP = {
    "joy":           {"primary": "joy",          "vi": "hạnh phúc",      "valence": 0.9,  "arousal": 0.7},
    "amusement":     {"primary": "joy",          "vi": "vui vẻ",         "valence": 0.8,  "arousal": 0.6},
    "excitement":    {"primary": "joy",          "vi": "hứng khởi",      "valence": 0.8,  "arousal": 0.9},
    "gratitude":     {"primary": "joy",          "vi": "biết ơn",        "valence": 0.85, "arousal": 0.4},
    "pride":         {"primary": "joy",          "vi": "tự hào",         "valence": 0.75, "arousal": 0.6},
    "relief":        {"primary": "joy",          "vi": "nhẹ nhõm",       "valence": 0.7,  "arousal": 0.3},
    "love":          {"primary": "love",         "vi": "yêu thương",     "valence": 0.95, "arousal": 0.5},
    "caring":        {"primary": "trust",        "vi": "quan tâm",       "valence": 0.7,  "arousal": 0.4},
    "optimism":      {"primary": "optimism",     "vi": "lạc quan",       "valence": 0.8,  "arousal": 0.6},
    "admiration":    {"primary": "trust",        "vi": "ngưỡng mộ",      "valence": 0.75, "arousal": 0.5},
    "desire":        {"primary": "anticipation", "vi": "khao khát",      "valence": 0.6,  "arousal": 0.7},
    "curiosity":     {"primary": "anticipation", "vi": "tò mò",          "valence": 0.5,  "arousal": 0.6},
    "sadness":       {"primary": "sadness",      "vi": "buồn bã",        "valence": -0.8, "arousal": 0.3},
    "grief":         {"primary": "sadness",      "vi": "đau buồn",       "valence": -0.9, "arousal": 0.2},
    "remorse":       {"primary": "remorse",      "vi": "hối hận",        "valence": -0.7, "arousal": 0.3},
    "disappointment":{"primary": "sadness",      "vi": "thất vọng",      "valence": -0.7, "arousal": 0.3},
    "embarrassment": {"primary": "sadness",      "vi": "xấu hổ",         "valence": -0.6, "arousal": 0.5},
    "anger":         {"primary": "anger",        "vi": "tức giận",       "valence": -0.8, "arousal": 0.9},
    "annoyance":     {"primary": "anger",        "vi": "khó chịu",       "valence": -0.6, "arousal": 0.6},
    "disapproval":   {"primary": "disapproval",  "vi": "phản đối",       "valence": -0.5, "arousal": 0.5},
    "fear":          {"primary": "fear",         "vi": "sợ hãi",         "valence": -0.8, "arousal": 0.8},
    "nervousness":   {"primary": "fear",         "vi": "lo lắng",        "valence": -0.6, "arousal": 0.7},
    "disgust":       {"primary": "disgust",      "vi": "ghê tởm",        "valence": -0.85,"arousal": 0.6},
    "surprise":      {"primary": "surprise",     "vi": "ngạc nhiên",     "valence": 0.1,  "arousal": 0.8},
    "confusion":     {"primary": "surprise",     "vi": "bối rối",        "valence": -0.3, "arousal": 0.5},
    "realization":   {"primary": "surprise",     "vi": "nhận ra",        "valence": 0.2,  "arousal": 0.5},
    "neutral":       {"primary": "neutral",      "vi": "bình thường",    "valence": 0.0,  "arousal": 0.0},
}

In [ ]:
rows = []
for item in ds['train']:
    labels = item['labels']
    if not labels:
        continue
    primary_label = id2label[labels[0]]
    emotion_info = GO_EMOTION_MAP.get(primary_label, GO_EMOTION_MAP['neutral'])
    secondary = [GO_EMOTION_MAP.get(id2label[l], {}).get("vi", "") for l in labels[1:]]
    
    rows.append({
        "text_en": item['text'],
        "primary_emotion": emotion_info["primary"],
        "primary_emotion_vi": emotion_info["vi"],
        "secondary_emotions": secondary,
        "valence": emotion_info["valence"],
        "arousal": emotion_info["arousal"],
        "source": "go_emotions",
    })

In [ ]:
df = pd.DataFrame(rows)
print(f"Total: {len(df)} samples")
df.to_csv("/kaggle/working/go_emotions_en_full.csv", index=False)
print("Saved")

In [ ]:
print("Loading Helsinki-NLP translation model")
model_name = "Helsinki-NLP/opus-mt-en-vi"

In [ ]:
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

In [ ]:
def translate_batch(texts, batch_size=64):
    translated = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        # Truncate dài quá
        batch = [t[:512] if len(t) > 512 else t for t in batch]
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128)
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translated.extend(decoded)
    
    return translated

In [ ]:
print(f"Translating {len(df)} samples...")
texts_en = df['text_en'].tolist()
texts_vi = translate_batch(texts_en, batch_size=64)

In [ ]:
df['text_vi'] = texts_vi
df.to_csv("/kaggle/working/go_emotions_vi_full.csv", index=False)
print("Translation complete")

In [ ]:
for i in range(5):
    print(f"\nEN: {df['text_en'][i]}")
    print(f"VI: {df['text_vi'][i]}")
    print(f"Emotion: {df['primary_emotion'][i]} ({df['primary_emotion_vi'][i]})")

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/go_emotions_vi_full.csv')